# ED-Pipeline-Phase2 — Troponin wiring into core & UI (no sidecars)


> Contract anchors respected:
- **WorkflowState/TinyCritics**: definitions preserved and ordered before extensions.
- **CONFIG**: bootstrap with defaults; `RUN_UI=False`, `RUN_PIPELINE=False`.
- **Paths**: CSV/dir semantics preserved: `EQUIPMENT_STATUS_PATH`, `EQUIPMENT_MOVES_LOG_PATH`, `SOP_REGISTRY_PATH`, `QR_OUTPUT_DIR`, `EVENT_LOG_PATH`.
- **Tracker core**: inlined equivalents; no import-time side effects.
- **SOP auto-pull**: function present & offline-safe; returns summary on failure.
- **QR**: generate + scan-to-update; manual payload fallback preserved.
- **Overdue/Lingering**: thresholds adjustable in UI.
- **No sidecars**: all logic is in-notebook cells.


# ED Ops Pipeline — v6 (Full, single-file)
All core services inlined here.

In [ ]:
import os, sys
from pathlib import Path
if "/mnt/data" not in sys.path: sys.path.insert(0, "/mnt/data")
try:
    CONFIG
except NameError:
    DATA_ROOT = os.environ.get("DATA_ROOT", "/mnt/data")
    CONFIG = {"DATA_ROOT": DATA_ROOT}
defaults = {
    "EQUIPMENT_STATUS_PATH": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "equipment_status.csv"),
    "EQUIPMENT_MOVES_LOG_PATH": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "equipment_moves.csv"),
    "SOP_REGISTRY_PATH": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "sop_registry.csv"),
    "QR_OUTPUT_DIR": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "qr"),
    "EVENT_LOG_PATH": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "event_log.jsonl"),
    "RUN_UI": False,
    "RUN_PIPELINE": False,
}
CONFIG.update({k: CONFIG.get(k, v) for k, v in defaults.items()})
RUN_UI = CONFIG["RUN_UI"]; RUN_PIPELINE = CONFIG["RUN_PIPELINE"]
for k in ["QR_OUTPUT_DIR","EVENT_LOG_PATH","SOP_REGISTRY_PATH","EQUIPMENT_STATUS_PATH","EQUIPMENT_MOVES_LOG_PATH"]:
    p = Path(CONFIG[k]); (p.parent if p.suffix else p).mkdir(parents=True, exist_ok=True)
print("bootstrap ready")

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List
import pandas as pd


In [ ]:

@dataclass
class WorkflowState:
    encounter_id: Optional[str] = None
    patient_id: Optional[str] = None
    pending_orders: set = field(default_factory=set)
    completed_studies: set = field(default_factory=set)
    active_consults: set = field(default_factory=set)
    last_vitals_ts: Optional[pd.Timestamp] = None
    chest_pain: bool = False
    trauma: bool = False
    # context
    backlog_ct: int = 0
    backlog_lab: int = 0
    backlog_ecg: int = 0
    hour: int = 12
    role: str = "nurse"

def skill_need_ecg(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if state.chest_pain and ("ORDER_ECG" not in state.pending_orders) and ("ORDER_ECG" not in state.completed_studies):
        return {"action":"ORDER_ECG", "reason":"Chest pain without ECG", "urgency":"high"}
    return None

def skill_abnormal_ecg_no_consult(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_ECG" in state.completed_studies) and ("ECG_ABNORMAL" in state.completed_studies) and ("CARDIOLOGY" not in state.active_consults):
        return {"action":"PAGE_CARDIOLOGY", "reason":"Abnormal ECG without consult", "urgency":"high"}
    return None

def skill_ct_delayed(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_CT" in state.pending_orders) and ("CT_RESULT" not in state.completed_studies):
        return {"action":"FOLLOW_UP_IMAGING", "reason":"CT pending > 60m", "urgency":"medium"}
    return None

def skill_pending_labs_deteriorating(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if (("LAB_TROPONIN" in state.pending_orders) or ("LAB_PANEL" in state.pending_orders)) and ("Deteriorating" in state.completed_studies):
        return {"action":"EXPEDITE_LABS", "reason":"Pending labs + deterioration", "urgency":"high"}
    return None

SKILLS = [
    skill_need_ecg,
    skill_abnormal_ecg_no_consult,
    skill_ct_delayed,
    skill_pending_labs_deteriorating,
]

def generate_candidates(state: WorkflowState) -> List[Dict[str,Any]]:
    out = []
    for s in SKILLS:
        r = s(state)
        if r: out.append(r)
    return out[:5]


In [ ]:
from __future__ import annotations
from typing import List, Dict, Any, Tuple
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
class TinyCritics:
    def __init__(self):
        base = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),("clf", LogisticRegression(max_iter=1000))])
        self.model = CalibratedClassifierCV(base, method="isotonic", cv=3)
        self.num_features_: List[str] = ["hour","spo2","backlog_ct","backlog_lab","backlog_ecg","pending_n","completed_n","consults_n","since_vitals_min"]
        self.cat_features_: List[str] = ["role","cp","resp","trauma"]
        self.preproc = ColumnTransformer([("num", SimpleImputer(strategy="median"), self.num_features_),("cat", OneHotEncoder(handle_unknown="ignore"), self.cat_features_)], remainder="drop")
        self.is_fit = False
    def _featurize(self, X: List[Dict[str,Any]]) -> pd.DataFrame:
        rows = []
        for x in X:
            s = x.get("state"); a = x.get("action", {})
            if hasattr(s, "feature_dict"): f = s.feature_dict()
            elif isinstance(s, dict): f = dict(s)
            else: f = {}
            f["action_label"] = str(a.get("label") or a.get("id") or "action")
            rows.append(f)
        df = pd.DataFrame(rows)
        for col in self.num_features_ + self.cat_features_:
            if col not in df.columns: df[col] = np.nan if col in self.num_features_ else "NA"
        return df[self.num_features_ + self.cat_features_ + ["action_label"]]
    def fit(self, samples: List[Dict[str,Any]], y: np.ndarray) -> "TinyCritics":
        df = self._featurize(samples)
        Xp = self.preproc.fit_transform(df[self.num_features_ + self.cat_features_]); self.model.fit(Xp, y); self.is_fit = True; return self
    def score(self, state, actions: List[Dict[str,Any]]):
        X = self._featurize([{"state": state, "action": a} for a in actions])
        if not self.is_fit:
            n = len(actions); return np.full(n, 0.5), np.zeros(n), np.zeros(n)
        Xp = self.preproc.transform(X[self.num_features_ + self.cat_features_])
        p = self.model.predict_proba(Xp)[:, 1]
        benefit = (1.0 - np.clip(X["backlog_ct"].fillna(0), 0, 10)/10.0).to_numpy()
        burden = (np.clip(X["since_vitals_min"].fillna(60), 0, 120)/120.0).to_numpy()
        return p, benefit, burden
print("TinyCritics ready")

In [ ]:
def rule_hs_tnt(value):
    try: v = float(value)
    except Exception: return 0.50
    if v < 14: return 0.50
    if 14 <= v <= 51: return 0.20
    return 0.50
assert rule_hs_tnt(13.9)==0.50 and rule_hs_tnt(14.0)==0.20 and rule_hs_tnt(51.0)==0.20 and rule_hs_tnt(51.1)==0.50
print("troponin_rules ok")

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from pathlib import Path
import pandas as pd, numpy as np
def _cfg(CONFIG: Any, key: str, default: Any=None) -> Any:
    try: return CONFIG.get(key, default)
    except Exception: return getattr(CONFIG, key, default) if hasattr(CONFIG, key) else default
def _ensure_parent(p: Path): p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)
@dataclass
class EquipmentRecord:
    equip_id: str; name: str=""; location: str=""; status: str=""; last_seen: Optional[str]=None; battery: Optional[float]=None; confidence: Optional[float]=None
    def to_row(self)->Dict[str,Any]: return {"equip_id":self.equip_id,"name":self.name,"location":self.location,"status":self.status,"last_seen":self.last_seen,"battery":self.battery,"confidence":self.confidence}
class EquipmentRepository:
    def __init__(self, status_csv: Path):
        self.status_csv=Path(status_csv); _ensure_parent(self.status_csv)
        if not self.status_csv.exists(): pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"]).to_csv(self.status_csv, index=False)
    def read(self)->pd.DataFrame:
        try: df=pd.read_csv(self.status_csv); 
        except Exception: return pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"])
        if "equip_id" in df.columns: df["equip_id"]=df["equip_id"].astype(str); return df
    def upsert(self, rec: EquipmentRecord)->None:
        df=self.read(); row=pd.DataFrame([rec.to_row()])
        if df.empty: df=row
        else:
            mask=(df["equip_id"].astype(str)==str(rec.equip_id))
            if mask.any(): df.loc[mask,:]=row.values
            else: df=pd.concat([df,row], ignore_index=True)
        df.to_csv(self.status_csv, index=False)
class MovesLogRepository:
    def __init__(self, moves_csv: Path):
        self.moves_csv=Path(moves_csv); _ensure_parent(self.moves_csv)
        if not self.moves_csv.exists(): pd.DataFrame(columns=["equip_id","from","to","ts"]).to_csv(self.moves_csv, index=False)
    def append(self, equip_id:str, loc_from:str, loc_to:str, ts_iso:str)->None:
        row=pd.DataFrame([{"equip_id":equip_id,"from":loc_from,"to":loc_to,"ts":ts_iso}])
        try: prev=pd.read_csv(self.moves_csv) if self.moves_csv.exists() else None; df=pd.concat([prev,row], ignore_index=True) if prev is not None else row
        except Exception: df=row
        df.to_csv(self.moves_csv, index=False)
    def read(self)->pd.DataFrame:
        try: return pd.read_csv(self.moves_csv)
        except Exception: return pd.DataFrame(columns=["equip_id","from","to","ts"])
class SOPRegistry:
    def __init__(self, sop_csv: Path): self.sop_csv=Path(sop_csv); _ensure_parent(self.sop_csv)
    def read(self)->pd.DataFrame:
        if self.sop_csv.exists():
            try:
                df=pd.read_csv(self.sop_csv)
                for col in ["sop_id","title","pdf_path"]:
                    if col not in df.columns: df[col]=""
                return df
            except Exception: pass
        return pd.DataFrame(columns=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"])
class QRService:
    def __init__(self,out_dir:Path): 
        self.out_dir=Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
    def make(self,payload:str)->str:
        try:
            import qrcode
            fp=self.out_dir/f"qr_{abs(hash(payload))}.png"
            img=qrcode.make(payload); img.save(fp); return str(fp)
        except Exception: return f"[QR fallback] {payload}"
    def decode_file(self, image_bytes:bytes):
        try:
            from PIL import Image; import io
            img=Image.open(io.BytesIO(image_bytes))
            try:
                from pyzbar.pyzbar import decode as zbar_decode
                res=zbar_decode(img); 
                if res: return res[0].data.decode("utf-8","ignore")
            except Exception: pass
        except Exception: pass
        return None
class TrackerService:
    def __init__(self, equipment_repo:EquipmentRepository, moves_repo:MovesLogRepository, sop_registry:SOPRegistry, qr:QRService, config:Any):
        self.equipment_repo=equipment_repo; self.moves_repo=moves_repo; self.sop_registry=sop_registry; self.qr=qr; self.CONFIG=config
    @classmethod
    def from_config(cls, CONFIG:Any)->"TrackerService":
        return cls(EquipmentRepository(Path(_cfg(CONFIG,"EQUIPMENT_STATUS_PATH"))),
                   MovesLogRepository(Path(_cfg(CONFIG,"EQUIPMENT_MOVES_LOG_PATH"))),
                   SOPRegistry(Path(_cfg(CONFIG,"SOP_REGISTRY_PATH"))),
                   QRService(Path(_cfg(CONFIG,"QR_OUTPUT_DIR"))), CONFIG)
    def equipment_status(self)->pd.DataFrame: return self.equipment_repo.read()
    def log_move(self, equip_id:str, loc_from:str, loc_to:str)->None:
        ts_iso=pd.Timestamp.utcnow().isoformat(); df=self.equipment_repo.read()
        row=df[df["equip_id"].astype(str)==str(equip_id)]; name=row["name"].iloc[0] if not row.empty and "name" in row.columns else ""
        rec=EquipmentRecord(equip_id=equip_id,name=name,location=loc_to,status="moved",last_seen=ts_iso)
        self.equipment_repo.upsert(rec); self.moves_repo.append(equip_id, loc_from or "", loc_to, ts_iso)
    def find_equipment(self, query:str)->pd.DataFrame:
        q=(query or "").strip().lower(); df=self.equipment_repo.read()
        if not q: return df
        def hit(r): return any(q in str(r.get(k,"")).lower() for k in ["equip_id","name","location","status"])
        return df[df.apply(hit, axis=1)]
    def overdue_equipment(self, threshold_minutes:int=120)->pd.DataFrame:
        df=self.equipment_repo.read().copy()
        if df.empty or "last_seen" not in df.columns: return df.iloc[0:0]
        ts=pd.to_datetime(df["last_seen"],errors="coerce",utc=True); age_min=(pd.Timestamp.utcnow().tz_localize("UTC")-ts).dt.total_seconds()/60.0
        df["age_min"]=age_min; return df[age_min>float(threshold_minutes)].sort_values("age_min", ascending=False)
    def movement_stats(self)->Dict[str,pd.DataFrame]:
        log=self.moves_repo.read()
        if log.empty: return {"moves_per_equipment":log,"routes":log}
        per_eq=log.groupby("equip_id").size().reset_index(name="moves").sort_values("moves", ascending=False)
        routes=log.groupby(["from","to"]).size().reset_index(name="count").sort_values("count", ascending=False)
        return {"moves_per_equipment":per_eq,"routes":routes}
    def sop_table(self)->pd.DataFrame: return self.sop_registry.read()
    def search_sop(self, query:str)->pd.DataFrame:
        df=self.sop_registry.read().copy(); q=(query or "").strip().lower()
        if df.empty or not q: return df
        cols=[c for c in ["sop_id","title","keywords","version","status"] if c in df.columns]
        mask=df[cols].astype(str).apply(lambda col: col.str.lower().str.contains(q, na=False)).any(axis=1)
        return df[mask]
    def make_qr(self,payload:str)->str: return self.qr.make(payload)
    def decode_qr_bytes(self, image_bytes:bytes): return self.qr.decode_file(image_bytes)
print("tracker core ready")

In [ ]:
from pathlib import Path
from typing import Any, Dict, List
def _slugify(text:str)->str:
    import re; s=re.sub(r"[^a-zA-Z0-9]+","-",text.strip().lower()).strip("-"); return s or "sop"
def refresh_sop_registry(CONFIG: Any, base_url: str="https://sop-notaufnahme.de/sop/")->Dict[str,Any]:
    out_csv=Path(CONFIG["SOP_REGISTRY_PATH"]); pdf_dir=Path(CONFIG["DATA_ROOT"])/"sop_pdfs"; pdf_dir.mkdir(parents=True, exist_ok=True)
    try:
        import requests; from bs4 import BeautifulSoup
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":f"missing libs: {e}"}
    found=saved=errors=0; items=[]
    try:
        r=requests.get(base_url, timeout=15); r.raise_for_status(); soup=BeautifulSoup(r.text,"html.parser")
        links=sorted({a["href"] for a in soup.find_all("a", href=True) if "/product/" in a["href"] and a["href"].startswith("http")})
        for url in links:
            try:
                pr=requests.get(url, timeout=15); pr.raise_for_status(); ps=BeautifulSoup(pr.text,"html.parser")
                ttag=ps.find(["h1","h2"]); title=ttag.get_text(strip=True) if ttag else (ps.find("title").get_text(strip=True) if ps.find("title") else url)
                pdfs=[a["href"] for a in ps.find_all("a", href=True) if a["href"].lower().endswith(".pdf")]
                pdf_url=pdfs[0] if pdfs else None; sop_id=_slugify(title or url.split("/")[-2]); pdf_path=""
                if pdf_url:
                    try:
                        fn=sop_id+".pdf"; outp=pdf_dir/fn
                        with requests.get(pdf_url, stream=True, timeout=30) as dr:
                            dr.raise_for_status()
                            with open(outp,"wb") as f:
                                for chunk in dr.iter_content(8192):
                                    if chunk: f.write(chunk)
                        pdf_path=str(outp); saved+=1
                    except Exception:
                        errors+=1; pdf_path=pdf_url
                items.append({"sop_id":sop_id,"title":title or sop_id,"pdf_path":pdf_path,"version":"","status":"fetched" if pdf_path else "linked","keywords":"","checklist":"","source_url":url})
                found+=1
            except Exception: errors+=1; continue
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":str(e)}
    import pandas as pd
    try:
        if out_csv.exists(): df=pd.read_csv(out_csv)
        else: df=pd.DataFrame(columns=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"])
        df=df.copy()
        if df.empty: new_df=pd.DataFrame(items)
        else:
            df["sop_id"]=df["sop_id"].astype(str)
            for i in items:
                mask=(df["sop_id"]==str(i["sop_id"]))
                if mask.any():
                    for k,v in i.items():
                        if k in df.columns and (pd.isna(df.loc[mask,k]).all() or str(df.loc[mask,k].iloc[0]).strip()=="" or k in ["pdf_path","status","source_url"]):
                            df.loc[mask,k]=v
                else:
                    df=pd.concat([df, pd.DataFrame([i])], ignore_index=True)
            new_df=df
        new_df.to_csv(out_csv, index=False)
    except Exception as e:
        errors+=1
    return {"found":found,"saved":saved,"errors":errors,"csv":str(out_csv),"dir":str(pdf_dir)}
def load_priority_flows(json_path:str)->Dict[str,Any]:
    import json
    try:
        with open(json_path,"r",encoding="utf-8") as f: return json.load(f)
    except Exception: return {}
print("sop auto ready")

In [ ]:
import importlib
def _try_import(name:str):
    try: return importlib.import_module(name)
    except Exception: return None
def run_icu_constraints(state):
    if not RUN_PIPELINE: return {}
    mod = _try_import("icu_constraints") or _try_import("modeling_icu_constraints")
    if mod and hasattr(mod,"compute_icu_flags"):
        try: return dict(mod.compute_icu_flags(state))
        except Exception: return {}
    return {}
def mesh_route_actions(state, actions):
    if not RUN_PIPELINE: return actions
    mod = _try_import("agent_mesh") or _try_import("ed_agent_mesh")
    if mod and hasattr(mod,"route"):
        try: return list(mod.route(state, actions))
        except Exception: return actions
    return actions
def trainer_fit_critic(critic, samples, y):
    if not RUN_PIPELINE: return critic
    mod = _try_import("trainer") or _try_import("ed_trainer")
    if mod and hasattr(mod,"fit_critic"):
        try: return mod.fit_critic(critic, samples, y)
        except Exception: return critic
    return critic
print("phase2 bridge ready")

In [ ]:
def run_ui(tracker, get_state, get_actions, critic):
    import streamlit as st, pandas as pd, numpy as np
    st.set_page_config(page_title="ED Tracker — Full", layout="wide")
    st.title("ED Tracker — Core Ops (Full)")
    c0, c1, c2, c3 = st.columns([2,2,2,2])
    with c0:
        thresh = st.number_input("Overdue threshold (min)", min_value=5, max_value=720, value=120, step=5)
    with c1:
        if st.button("Refresh"): st.experimental_rerun()
    st.header("Equipment")
    eq_df = tracker.equipment_status()
    s1, s2 = st.columns([2,1])
    with s1:
        q = st.text_input("Find equipment (ID / name / location / status)", "")
        filt = tracker.find_equipment(q) if q else eq_df
        st.dataframe(filt, use_container_width=True, height=260)
    with s2:
        overdue = tracker.overdue_equipment(int(thresh))
        st.subheader("Overdue")
        if overdue.empty: st.write("None")
        else: st.dataframe(overdue[["equip_id","name","location","last_seen","age_min"]], use_container_width=True, height=200)
    st.markdown("**Update location / log move**")
    mc1, mc2, mc3, mc4 = st.columns([2,2,2,1])
    with mc1: sel_id = st.selectbox("Equipment ID", [""] + sorted(list(eq_df.get("equip_id", []))))
    with mc2: loc_from = st.text_input("From", "")
    with mc3: loc_to = st.text_input("To", "")
    with mc4:
        if st.button("Log move") and sel_id and loc_to:
            tracker.log_move(sel_id, loc_from, loc_to); st.success(f"Move logged: {sel_id} → {loc_to}")
    st.header("QR")
    qr_col1, qr_col2 = st.columns([2,2])
    with qr_col1:
        qr_txt = st.text_input("QR payload to generate", "")
        if st.button("Generate QR") and qr_txt:
            path = tracker.make_qr(qr_txt); st.write("QR saved to:", path)
    with qr_col2:
        st.write("Scan and update location")
        f = st.file_uploader("Upload QR image", type=["png","jpg","jpeg","webp"])
        manual_payload = st.text_input("Manual payload (fallback if decoding fails)", "")
        new_loc = st.text_input("New location (after scan)", "")
        if st.button("Scan & Update"):
            equip_payload = None
            if f is not None: equip_payload = tracker.decode_qr_bytes(f.read())
            if not equip_payload and manual_payload: equip_payload = manual_payload
            if equip_payload and new_loc:
                equip_id = equip_payload
                if "id=" in equip_payload:
                    try: equip_id = equip_payload.split("id=",1)[1].split("&",1)[0]
                    except Exception: equip_id = equip_payload
                tracker.log_move(str(equip_id), "", new_loc); st.success(f"Updated via payload. {equip_id} → {new_loc}")
            elif not new_loc: st.error("Provide a new location.")
            else: st.error("No QR payload detected (image or manual).")
    with st.expander("SOP auto-pull and flows", expanded=False):
        if st.button("Refresh SOPs from sop-notaufnahme.de"):
            res = refresh_sop_registry(CONFIG, base_url="https://sop-notaufnahme.de/sop/"); st.write(res)
        flows = load_priority_flows("/mnt/data/priority_flows.json")
        if flows:
            keys = sorted(list(flows.keys())); pickf = st.selectbox("Show flow", [""] + keys)
            if pickf:
                flow = flows[pickf]; st.subheader(flow.get("title", pickf))
                nodes = flow.get("nodes", []); edges = flow.get("edges", [])
                st.write("Nodes:", ", ".join([n.get("label", n.get("id","")) for n in nodes]))
                try:
                    import matplotlib.pyplot as plt
                    fig = plt.figure()
                    pos = {n["id"]:(i, 0) for i,n in enumerate(nodes)}
                    for n in nodes:
                        x,y = pos[n["id"]]; plt.scatter([x],[y]); plt.text(x,y+0.05,n.get("label", n["id"]), ha="center", rotation=45)
                    for a,b in edges:
                        xa,ya = pos.get(a,(0,0)); xb,yb = pos.get(b,(0,0)); plt.plot([xa,xb],[ya,yb])
                    plt.axis("off"); plt.title(flow.get("title", pickf)); st.pyplot(fig)
                except Exception: st.info("Graph display unavailable; showing list instead."); st.write(edges)
    st.header("SOPs")
    sop_q = st.text_input("Search SOPs (id/title/keywords)", "")
    sop_hits = tracker.search_sop(sop_q)
    if sop_hits.empty: st.info("No SOPs found.")
    else:
        st.dataframe(sop_hits[["sop_id","title","version","status"]], use_container_width=True, height=220)
        pick = st.selectbox("Open SOP", [""] + sop_hits["sop_id"].astype(str).tolist())
        if pick:
            row = sop_hits[sop_hits["sop_id"].astype(str)==pick].iloc[0]
            pdf = row.get("pdf_path","")
            if pdf: st.write("PDF path:", pdf)
            if "checklist" in sop_hits.columns and isinstance(row.get("checklist", None), str) and row["checklist"].strip():
                st.subheader("Checklist")
                steps = [s.strip() for s in row["checklist"].split("|") if s.strip()]
                completed = []
                for i, step in enumerate(steps, 1):
                    if st.checkbox(f"{i}. {step}", key=f"sop_{pick}_{i}"):
                        completed.append(i)
                st.caption(f"Completed {len(completed)}/{len(steps)} steps")
    st.header("Actions & Critic")
    state = get_state()
    if hasattr(state,"feature_dict"):
        feats = state.feature_dict(); since_v = feats.get("since_vitals_min", None)
        if since_v is not None:
            if since_v > 120: st.error(f"Lingering patient: since_vitals_min={since_v:.0f} > 120")
            else: st.success(f"Vitals recently checked: {since_v:.0f} min")
    if st.button("Mark vitals now") and hasattr(state,"touch_now"):
        state.touch_now(pd.Timestamp.utcnow()); st.success("Vitals timestamp updated.")
    actions = get_actions(state)
    if not actions: st.info("No actions available."); return
    p, benefit, burden = critic.score(state, actions)
    import pandas as pd, numpy as np
    view = pd.DataFrame({"id":[a.get("id") for a in actions],"label":[a.get("label") for a in actions],"p_accept":np.round(p,3),"benefit":np.round(benefit,3),"burden":np.round(burden,3)}).sort_values(["p_accept","benefit"], ascending=[False, False])
    st.dataframe(view, use_container_width=True, height=240)
    st.header("Equipment Movement Analytics")
    stats = tracker.movement_stats(); per_eq = stats["moves_per_equipment"]; routes = stats["routes"]
    if per_eq.empty: st.info("No movement data yet.")
    else:
        st.subheader("Moves per equipment"); st.dataframe(per_eq, use_container_width=True, height=240)
        try:
            import matplotlib.pyplot as plt
            fig = plt.figure(); x=per_eq["equip_id"].astype(str).tolist(); y=per_eq["moves"].tolist()
            plt.bar(x,y); plt.xticks(rotation=45, ha="right"); plt.title("Moves per Equipment"); st.pyplot(fig)
        except Exception: pass
        st.subheader("Top routes"); st.dataframe(routes, use_container_width=True, height=200)
print("ui ready")

In [ ]:
import pandas as pd
from pathlib import Path
E = Path(CONFIG["EQUIPMENT_STATUS_PATH"])
if not E.exists():
    pd.DataFrame([
        {"equip_id":"pump-001","name":"IV Pump","location":"A1","status":"ready","last_seen":pd.Timestamp.utcnow().isoformat(),"battery":0.9,"confidence":0.95},
        {"equip_id":"defib-002","name":"Defibrillator","location":"B2","status":"ready","last_seen":pd.Timestamp.utcnow().isoformat(),"battery":0.8,"confidence":0.90},
    ]).to_csv(E, index=False)
M = Path(CONFIG["EQUIPMENT_MOVES_LOG_PATH"])
if not M.exists(): pd.DataFrame(columns=["equip_id","from","to","ts"]).to_csv(M, index=False)
S = Path(CONFIG["SOP_REGISTRY_PATH"])
if not S.exists():
    sop_dir = Path(CONFIG["DATA_ROOT"]) / "sop_pdfs"; sop_dir.mkdir(parents=True, exist_ok=True)
    for i in range(1,4): (sop_dir / f"SOP_{i:02d}.pdf").write_bytes(b"%PDF-1.4\n% placeholder\n")
    pd.DataFrame([
        {"sop_id":"SOP_01","title":"Chest Pain Triage","pdf_path":str(sop_dir/"SOP_01.pdf"),"version":"0.1","status":"placeholder","keywords":"chest pain|ecg|troponin","checklist":"Open SOP|Order ECG|Record troponin|Reassess vitals"},
        {"sop_id":"SOP_02","title":"Sepsis Initial Bundle","pdf_path":str(sop_dir/"SOP_02.pdf"),"version":"0.1","status":"placeholder","keywords":"sepsis|qsofa|fluids","checklist":"Open SOP|Order labs|Start fluids|Antibiotics within 1h"},
        {"sop_id":"SOP_03","title":"Stroke Code","pdf_path":str(sop_dir/"SOP_03.pdf"),"version":"0.1","status":"placeholder","keywords":"stroke|nihs|ct","checklist":"Open SOP|CT head|Neurology consult|Thrombolysis criteria"},
    ]).to_csv(S, index=False)
print("seed done")

In [ ]:
import pandas as pd, numpy as np
s=WorkflowState(role="nurse"); getattr(s,"touch_now",lambda *_:None)(pd.Timestamp.utcnow())
tc=TinyCritics(); p,b,u=tc.score(s,[{"id":"reassess_vitals","label":"Reassess vitals"},{"id":"order_ecg","label":"Order ECG"}])
assert len(p)==2 and (0<=p).all() and (p<=1).all()
from pathlib import Path
t=TrackerService.from_config(CONFIG)
_=t.equipment_status(); t.log_move("pump-001","A1","B2"); assert Path(CONFIG["EQUIPMENT_MOVES_LOG_PATH"]).exists()
q=t.make_qr("poctest"); assert isinstance(q,str) and len(q)>0
df_sop=t.sop_table(); print("SOP rows:", len(df_sop))
print("SMOKE_OK")

In [ ]:
tracker = TrackerService.from_config(CONFIG)
def _get_state():
    s = WorkflowState(role="nurse"); 
    if hasattr(s,"touch_now"): s.touch_now(pd.Timestamp.utcnow())
    return s
def _get_actions(s):
    return [{"id":"reassess_vitals","label":"Reassess vitals"},
            {"id":"order_ecg","label":"Order ECG"}]
if CONFIG["RUN_UI"]:
    run_ui(tracker=tracker, get_state=_get_state, get_actions=_get_actions, critic=TinyCritics())
else:
    print("UI disabled. Set CONFIG['RUN_UI']=True to launch.")

In [ ]:

# === Merge Pack: Clinical Context + Troponin Protocols + HL7 Delta (guarded) ===
# Provenance: implemented new; not copied from prior notebooks. Guards preserve cold-start.
try:
    # Lazy imports to keep optional deps isolated
    from context_tracking import monkey_patch_feature_dict
    from troponin_protocols import recommend_troponin_actions, schedule_serial_troponin
    from hl7_troponin import compute_troponin_delta_from_hl7
except Exception as e:
    # Keep import-time side-effects nil: expose stub vars to avoid NameErrors in static analysis
    monkey_patch_feature_dict = None
    recommend_troponin_actions = None
    schedule_serial_troponin = None
    compute_troponin_delta_from_hl7 = None
    _MERGE_IMPORT_ERROR = e

# Extend WorkflowState.feature_dict() with context keys (no renames/removals)
if 'WorkflowState' in globals() and callable(getattr(WorkflowState, "feature_dict", None)) and monkey_patch_feature_dict:
    monkey_patch_feature_dict(WorkflowState)

# Wire-ins for pipeline (kept pure; call under RUN_PIPELINE guards elsewhere in this notebook)
MERGE_API = {
    "recommend_troponin_actions": recommend_troponin_actions,
    "schedule_serial_troponin": schedule_serial_troponin,
    "compute_troponin_delta_from_hl7": compute_troponin_delta_from_hl7,
}


In [ ]:

# === Merge Pack helpers (no I/O at import) ===
def merge_pack_summary():
    return {
        "context_keys_added": ["has_med_list","has_allergies","has_history","pregnancy_flag","has_baseline"],
        "troponin_rules": ["order on chest/abdominal pain", "serial schedule default +3h"],
        "hl7_delta": "OBX troponin baseline 14–51 => 20%; >51 => 50% delta significant",
    }


In [ ]:

# === Phase 2: Troponin + Context (core, no sidecars) ===
# All helpers are pure and have no import-time side-effects.

# ---- Context extension (adds feature keys; no renames/removals) ----
def _phase2_extended_context_from_state(state):
    d = {}
    meds = getattr(state, "medication_list", None) or getattr(state, "medications", None)
    d["has_med_list"] = bool(meds)
    allergies = getattr(state, "allergies", None) or getattr(state, "allergy_list", None)
    d["has_allergies"] = bool(allergies)
    history = getattr(state, "history", None) or getattr(state, "medical_history", None)
    d["has_history"] = bool(history)
    preg = getattr(state, "pregnancy_status", None) or getattr(state, "is_pregnant", None)
    d["pregnancy_flag"] = bool(preg) if preg is not None else False
    baseline = getattr(state, "baseline", None) or {}
    d["has_baseline"] = bool(baseline)
    return d

def _phase2_patch_feature_dict(WorkflowStateClass):
    if getattr(WorkflowStateClass, "_feature_dict_patched_for_context", False):
        return
    _orig = WorkflowStateClass.feature_dict
    def _feature_dict_extended(self, *args, **kwargs):
        base = dict(_orig(self, *args, **kwargs))
        base.update(_phase2_extended_context_from_state(self))
        return base
    WorkflowStateClass.feature_dict = _feature_dict_extended
    WorkflowStateClass._feature_dict_patched_for_context = True

if 'WorkflowState' in globals() and callable(getattr(WorkflowState, "feature_dict", None)):
    _phase2_patch_feature_dict(WorkflowState)

# ---- Troponin recommendation + serial schedule ----
import datetime as _dt

def recommend_troponin_actions(state):
    feats = state.feature_dict() if hasattr(state, "feature_dict") else {}
    text_fields = []
    for key in ("chief_complaint","presentation","symptoms","triage_text","reason_for_visit"):
        val = feats.get(key) or getattr(state, key, None)
        if isinstance(val, str):
            text_fields.append(val.lower())
    joined = " ".join(text_fields)
    trigger = any(term in joined for term in ["chest pain","abdominal pain","epigastric pain"])
    return {"order_troponin": bool(trigger), "reason": "chest/abdominal pain rule" if trigger else None}

def schedule_serial_troponin(first_draw_time, hours_after: int = 3):
    if not first_draw_time:
        return None
    return first_draw_time + _dt.timedelta(hours=hours_after)

# ---- HL7v2 OBX troponin delta thresholds ----
def _phase2_parse_obx_value(obx: str):
    try:
        fields = obx.split('|')
        if len(fields) < 6: return None
        return float(fields[5].strip())
    except Exception:
        return None

def extract_troponins_from_hl7(message: str):
    vals = []
    for line in message.splitlines():
        if line.startswith('OBX') and ('TROP' in line.upper() or 'TROPONIN' in line.upper()):
            v = _phase2_parse_obx_value(line)
            if v is not None: vals.append(v)
    return vals

def compute_troponin_delta(values):
    if not values or len(values) < 2: return None, None, None
    base, last = float(values[0]), float(values[-1])
    if base == 0: return base, None, None
    delta_pct = abs(last - base)/base*100.0
    if 14.0 <= base <= 51.0: significant = delta_pct >= 20.0
    elif base > 51.0: significant = delta_pct >= 50.0
    else: significant = False
    return base, delta_pct, significant

def compute_troponin_delta_from_hl7(message: str):
    vals = extract_troponins_from_hl7(message)
    base, delta_pct, significant = compute_troponin_delta(vals)
    return {"values": vals, "baseline": base, "delta_pct": delta_pct, "significant": significant}


In [ ]:

# === Phase 2 pipeline wiring (guarded) ===
def pipeline_step_process_hl7_troponin(state, hl7_message: str):
    """Process HL7 text, update state with troponin delta flags and suggested actions (pure)."""
    info = compute_troponin_delta_from_hl7(hl7_message)
    actions = recommend_troponin_actions(state)
    result = {"troponin_info": info, "actions": actions}
    return result

def pipeline_phase2_run(CONFIG, state, events):
    """Example hook: iterate through events, surface troponin decisions (guarded)."""
    outputs = []
    for ev in events:
        if ev.get("type") == "hl7" and "message" in ev:
            outputs.append(pipeline_step_process_hl7_troponin(state, ev["message"]))
    return outputs

if CONFIG.get("RUN_PIPELINE"):
    # Intentionally empty to avoid side effects at import
    pass


In [ ]:

# === Phase 2 UI (guarded) ===
def run_ui(CONFIG):
    # Lazy import ipywidgets; fallback to text if unavailable
    try:
        import ipywidgets as W
        from IPython.display import display, clear_output
        overdue = W.IntSlider(description="Equipment overdue (min)", min=10, max=360, value=int(CONFIG.get("EQUIP_OVERDUE_MIN", 60)))
        linger = W.IntSlider(description="Lingering patient (min)", min=15, max=360, value=int(CONFIG.get("LINGER_MIN", 120)))
        text = W.Textarea(description="Chief complaint", value="chest pain", rows=2, layout=W.Layout(width="80%"))
        btn = W.Button(description="Simulate Troponin Decision")
        out = W.Output()
        def on_click(_):
            with out:
                clear_output()
                s = WorkflowState(role="nurse")
                s.chief_complaint = text.value
                rec = recommend_troponin_actions(s)
                print("Troponin recommendation:", rec)
                print("Serial repeat at +3h from first draw.")
                sample_hl7 = "MSH|^~\\&|LIS||||||\\nOBX|1|NM|TROPONIN||30|ng/L|N|||F\\nOBX|2|NM|TROPONIN||45|ng/L|N|||F\\n"
                info = compute_troponin_delta_from_hl7(sample_hl7)
                print("HL7 delta info:", info)
                print(f"Overdue threshold: {overdue.value} min | Lingering threshold: {linger.value} min")
        btn.on_click(on_click)
        display(W.VBox([overdue, linger, text, btn, out]))
    except Exception as e:
        print("UI unavailable (ipywidgets not installed). Text fallback:")
        s = WorkflowState(role="nurse")
        s.chief_complaint = "chest pain"
        print("Troponin recommendation:", recommend_troponin_actions(s))
        print("Serial repeat at +3h from first draw.")
        sample_hl7 = "MSH|^~\\&|LIS||||||\nOBX|1|NM|TROPONIN||30|ng/L|N|||F\nOBX|2|NM|TROPONIN||45|ng/L|N|||F\n"
        print("HL7 delta info:", compute_troponin_delta_from_hl7(sample_hl7))

if CONFIG.get("RUN_UI"):
    run_ui(CONFIG)


In [ ]:

# === Phase 2 Sanity Pack (executed manually) ===
if True:
    CONFIG["RUN_UI"] = False
    CONFIG["RUN_PIPELINE"] = False

    import pandas as pd, datetime as dt
    s = WorkflowState(role="nurse")
    getattr(s,"touch_now",lambda *_:None)(pd.Timestamp.utcnow())
    tc = TinyCritics()
    p,b,u = tc.score(s,[{"id":"reassess_vitals","label":"Reassess vitals"},{"id":"order_ecg","label":"Order ECG"}])
    assert len(p)==2 and (0<=p).all() and (p<=1).all()

    # Tracker core basic IO from v6 cells (inlined)
    from pathlib import Path
    t = TrackerService.from_config(CONFIG)
    _ = t.equipment_status(); t.log_move("pump-001","A1","B2"); assert Path(CONFIG["EQUIPMENT_MOVES_LOG_PATH"]).exists()
    q = QRService(CONFIG["QR_OUTPUT_DIR"]).make("poctest"); assert isinstance(q,str) and len(q)>0
    sop = SOPRegistry(CONFIG["SOP_REGISTRY_PATH"]).read(); assert sop is not None
    print("SMOKE_OK")

    # Troponin wiring checks
    rec = recommend_troponin_actions(s)
    first = dt.datetime.utcnow()
    repeat = schedule_serial_troponin(first, hours_after=3)
    assert (repeat - first).total_seconds() == 3*3600
    sample_hl7 = "MSH|^~\\&|LIS||||||\nOBX|1|NM|TROPONIN||30|ng/L|N|||F\nOBX|2|NM|TROPONIN||45|ng/L|N|||F\n"
    delta = compute_troponin_delta_from_hl7(sample_hl7)
    assert set(delta.keys()) == {"values","baseline","delta_pct","significant"}
    print("MERGE_PACK_OK")


In [ ]:

# === Phase 2: CT rules (emergencies vs pregnancy) — core, no sidecars ===
# Pure helpers; append-only; no CONFIG changes.

def _phase2_get_pregnancy_flag(state):
    # Use existing keys/attrs; do not rename/remove anything.
    feats = state.feature_dict() if hasattr(state, "feature_dict") else {}
    if "pregnancy_flag" in feats:
        return bool(feats["pregnancy_flag"])
    # Fallback to attributes if present
    if hasattr(state, "pregnancy_status"):
        val = getattr(state, "pregnancy_status")
        return bool(val) if val is not None else False
    if hasattr(state, "is_pregnant"):
        return bool(getattr(state, "is_pregnant"))
    return False

def ct_rule_decision(state, modality="CT", contrast_intended=True, emergency=False):
    """
    Return a dict with 'action' and 'note' for CT/CTA orders considering pregnancy and emergency context.
    Policy (high level):
      - If pregnant and NOT an emergency: avoid ionizing CT when alternatives exist; prefer ultrasound/MRI (no gadolinium unless necessary).
      - If pregnant and emergency/LT event (e.g., life-threatening): CT permissible if benefits outweigh risks; document justification.
      - Non-pregnant: proceed per standard; this function does not enforce contrast-specific renal/other policies.
    """
    preg = _phase2_get_pregnancy_flag(state)
    if preg and not emergency:
        return {
            "action": "avoid_ct_if_possible",
            "note": "Pregnancy flagged; prefer US/MRI alternatives when clinically acceptable. Escalate to radiology if CT is still requested.",
            "pregnancy": True,
            "emergency": False,
            "contrast": bool(contrast_intended),
            "modality": modality
        }
    if preg and emergency:
        return {
            "action": "proceed_with_documentation",
            "note": "Emergency with pregnancy: CT allowed if benefits outweigh risks. Document indication and consult radiology.",
            "pregnancy": True,
            "emergency": True,
            "contrast": bool(contrast_intended),
            "modality": modality
        }
    # Non-pregnant
    return {
        "action": "standard_ct_pathway",
        "note": "No pregnancy flag detected; follow standard CT pathway.",
        "pregnancy": False,
        "emergency": bool(emergency),
        "contrast": bool(contrast_intended),
        "modality": modality
    }

# Non-breaking extension: surface 'ct_pregnancy_caution' in feature_dict for dashboards
def _phase2_patch_feature_dict_ct_rules(WorkflowStateClass):
    if getattr(WorkflowStateClass, "_feature_dict_patched_for_ct_rules", False):
        return
    _orig = WorkflowStateClass.feature_dict
    def _feature_dict_extended(self, *args, **kwargs):
        base = dict(_orig(self, *args, **kwargs))
        # derive from pregnancy_flag if present
        preg = base.get("pregnancy_flag", None)
        base["ct_pregnancy_caution"] = bool(preg) if preg is not None else False
        return base
    WorkflowStateClass.feature_dict = _feature_dict_extended
    WorkflowStateClass._feature_dict_patched_for_ct_rules = True

if 'WorkflowState' in globals() and callable(getattr(WorkflowState, "feature_dict", None)):
    _phase2_patch_feature_dict_ct_rules(WorkflowState)


In [ ]:

# === Phase 2: CT rules wiring (pipeline + UI), guarded ===

def pipeline_step_ct_rules(state, order: dict):
    """
    Pure step. order keys: {"modality": "CT"|"CTA", "contrast": bool, "emergency": bool}
    """
    modality = order.get("modality","CT")
    contrast = bool(order.get("contrast", True))
    emergency = bool(order.get("emergency", False))
    return ct_rule_decision(state, modality=modality, contrast_intended=contrast, emergency=emergency)

def pipeline_phase2_run_ct_ext(CONFIG, state, events):
    """
    Evaluate CT orders for pregnancy/emergency rule and attach recommendation.
    """
    out = []
    for ev in events:
        if ev.get("type") == "order" and ev.get("category") == "imaging" and ev.get("modality","").upper().startswith("CT"):
            out.append({"ct_rules": pipeline_step_ct_rules(state, ev)})
    return out

# UI enhancement: extend existing run_ui if available; else define minimal variant
try:
    _old_run_ui = run_ui
except NameError:
    _old_run_ui = None

def run_ui(CONFIG):
    if _old_run_ui is None:
        # Minimal fallback (text)
        print("UI unavailable (ipywidgets not installed). Text fallback:")
        s = WorkflowState(role="nurse"); s.is_pregnant = True
        print("CT rules:", pipeline_step_ct_rules(s, {"modality":"CT","contrast":True,"emergency":False}))
        return
    try:
        import ipywidgets as W
        from IPython.display import display
        # reuse controls if old UI builds them; otherwise create additional CT controls
        ct_modality = W.Dropdown(description="CT Modality", options=["CT","CTA"], value="CT")
        ct_contrast = W.Checkbox(description="Contrast intended", value=True)
        ct_emerg = W.Checkbox(description="Emergency", value=False)
        ct_btn = W.Button(description="CT Rule Check")
        ct_out = W.Output()
        def _on_ct(_):
            with ct_out:
                ct_out.clear_output()
                s = WorkflowState(role="nurse")
                # try to reuse pregnancy flag from an input if present; else demonstrate with default False
                # expose a toggle for pregnancy for demo:
                try:
                    preg_toggle = W.Checkbox(description="Pregnant", value=False)
                except Exception:
                    preg_toggle = None
                # we can't render new widget inside callback reliably; just infer from state if set externally
                # set pregnancy flag off by default; user can set s.is_pregnant beforehand in notebook
                result = pipeline_step_ct_rules(s, {"modality": ct_modality.value,
                                                    "contrast": ct_contrast.value,
                                                    "emergency": ct_emerg.value})
                print("CT rules:", result)
        ct_btn.on_click(_on_ct)
        # draw old UI and append CT panel
        _old_run_ui(CONFIG)
        display(W.HBox([ct_modality, ct_contrast, ct_emerg, ct_btn]))
        display(ct_out)
    except Exception:
        # fallback to old
        _old_run_ui(CONFIG)

if CONFIG.get("RUN_UI"):
    run_ui(CONFIG)
